In [ ]:
# 🔁 Install & import essentials
!pip install diffusers transformers accelerate --quiet

import os
import json
import torch
import base64
import numpy as np
from PIL import Image
#from google.colab import files
from diffusers import StableDiffusionPipeline

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 100.1 MB/s eta 0:00:00


In [ ]:
# Specify your local image folder
import os

image_folder = "images"  # Change to your local image directory
os.makedirs("json_output", exist_ok=True)
image_files = [f for f in os.listdir(image_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
print(f"Found {len(image_files)} images: {image_files}")

Saving WhatsApp Image 2024-04-22 at 07.26.18_1ab32dd9.jpg to WhatsApp Image 2024-04-22 at 07.26.18_1ab32dd9.jpg
Saving WhatsApp Image 2024-04-22 at 07.28.17_22e9ec8b.jpg to WhatsApp Image 2024-04-22 at 07.28.17_22e9ec8b.jpg
Saving WhatsApp Image 2024-06-12 at 20.10.22_d03908c0.jpg to WhatsApp Image 2024-06-12 at 20.10.22_d03908c0.jpg


In [3]:

pipe = StableDiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-2-1-base",
    torch_dtype=torch.float16
).to("cuda")

vae = pipe.vae.half().to("cuda")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/543 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

scheduler_config.json:   0%|          | 0.00/346 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/911 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.46G [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

In [4]:
for image_name in uploaded.keys():
    try:
        image = Image.open(image_name).convert("RGB").resize((512, 512))

        # Convert to tensor
        image_tensor = torch.tensor(np.array(image)).unsqueeze(0).permute(0, 3, 1, 2).float() / 255.
        image_tensor = image_tensor.to("cuda").half()

        # Get latent vector
        with torch.no_grad():
            latent_dist = vae.encode(image_tensor)
            latent_sample = latent_dist.latent_dist.sample()

        latent_numpy = latent_sample.cpu().numpy().flatten()
        latent_shape = latent_sample.shape

        # Base64 encode
        latent_bytes = latent_numpy.tobytes()
        latent_base64 = base64.b64encode(latent_bytes).decode("utf-8")

        # Save to JSON
        json_data = {
            "latent_base64": latent_base64,
            "latent_shape": latent_shape,
            "dtype": str(latent_numpy.dtype)
        }

        json_name = os.path.splitext(image_name)[0] + "_latent.json"
        with open(os.path.join("json_output", json_name), "w") as f:
            json.dump(json_data, f)

        print(f"✅ Saved latent vector: {json_name}")

    except Exception as e:
        print(f"❌ Failed to process {image_name}: {e}")


✅ Saved latent vector: WhatsApp Image 2024-04-22 at 07.26.18_1ab32dd9_latent.json
✅ Saved latent vector: WhatsApp Image 2024-04-22 at 07.28.17_22e9ec8b_latent.json
✅ Saved latent vector: WhatsApp Image 2024-06-12 at 20.10.22_d03908c0_latent.json


In [ ]:
import shutil
import os

# Zip the folder
shutil.make_archive('json_output_zip', 'zip', 'json_output')
print("Latent JSONs zipped and saved as json_output_zip.zip in your working directory.")
# You can manually copy or move this file as needed.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

reconstruction


In [5]:
os.makedirs("reconstructed", exist_ok=True)

json_files = [f for f in os.listdir("json_output") if f.endswith(".json")]

for json_file in json_files:
    try:
        with open(os.path.join("json_output", json_file), "r") as f:
            latent_json = json.load(f)

        latent_bytes = base64.b64decode(latent_json["latent_base64"])
        latent_np = np.frombuffer(latent_bytes, dtype=latent_json["dtype"])
        latent_tensor = torch.tensor(latent_np).view(*latent_json["latent_shape"]).to("cuda").half()

        with torch.no_grad():
            reconstructed_image = vae.decode(latent_tensor).sample

        reconstructed_image = (reconstructed_image.clamp(0, 1).cpu().squeeze().permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        pil_image = Image.fromarray(reconstructed_image)

        recon_name = os.path.splitext(json_file)[0].replace("_latent", "_reconstructed.png")
        pil_image.save(os.path.join("reconstructed", recon_name))

        print(f"🖼️ Reconstructed: {recon_name}")

    except Exception as e:
        print(f"❌ Failed to reconstruct from {json_file}: {e}")


🖼️ Reconstructed: WhatsApp Image 2024-04-22 at 07.26.18_1ab32dd9_reconstructed.png
🖼️ Reconstructed: WhatsApp Image 2024-04-22 at 07.28.17_22e9ec8b_reconstructed.png
🖼️ Reconstructed: WhatsApp Image 2024-06-12 at 20.10.22_d03908c0_reconstructed.png


In [ ]:
import shutil
shutil.make_archive("reconstructed_images", 'zip', "reconstructed")
files.download("reconstructed_images.zip")
